In [ ]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import pandas as pd
import os
import glob

class enter_lable:
    def __init__(self, root, img_dir, excel_path):
        self.root = root
        self.img_dir = img_dir
        self.excel_path = excel_path
        self.img_files = sorted(glob.glob(os.path.join(self.img_dir, '**', '*.jpg'), recursive=True))
        self.img_index = 0
        self.df = pd.read_csv(self.excel_path)
        # GUI components
        self.img_label = tk.Label(self.root)
        self.img_label.pack()

        self.entry_fields = {}
        self.entries_frame = tk.Frame(self.root)
        self.entries_frame.pack()

        # Create input fields for each specified column
        columns = ['filter', 'rell_number', 'date', 'hour', 'minute', 'seconds']
        for i, col in enumerate(columns):
            label = tk.Label(self.entries_frame, text=f'{col}:')
            label.grid(row=i, column=0)
            entry = tk.Entry(self.entries_frame)
            entry.grid(row=i, column=1)
            self.entry_fields[col] = entry

        self.index_entry = tk.Entry(self.root)
        self.index_entry.pack()

        self.go_button = tk.Button(self.root, text="Go to Image", command=self.goto_image)
        self.go_button.pack()

        self.next_button = tk.Button(self.root, text="Next Image", command=self.next_image)
        self.next_button.pack()

        self.previous_button = tk.Button(self.root, text="Previous Image", command=self.previous_image)
        self.previous_button.pack()

        self.save_button = tk.Button(self.root, text="Save", command=self.save_data)
        self.save_button.pack()
        self.df = pd.read_csv(self.excel_path)
        for index, img_file in enumerate(self.img_files):
            img_name = os.path.basename(img_file).split('.')[0]
            if self.df[self.df['ID'] == img_name].empty or self.df[self.df['ID'] == img_name].isnull().values.any():
                self.img_index = index
                break

        self.show_image()

    def show_image(self):
        while self.img_index < len(self.img_files):
            img_path = self.img_files[self.img_index]

            img_name = os.path.basename(self.img_files[self.img_index]).split('.')[0]
            # Check if this image already has complete data
            if not self.df[self.df['ID'] == img_name].isnull().values.any():
                self.img_index += 1  # Skip to the next image
                if self.img_index >= len(self.img_files):
                    tk.messagebox.showinfo("End", "No more images.")
                    return
                continue  # Continue checking the next image

            img = Image.open(img_path)

            img_cropped = img.crop((1800, 0, img.width, img.height))

            new_width = img_cropped.width * 1
            new_height = img_cropped.height * 1
            img_resized = img_cropped.resize((new_width, new_height), Image.LANCZOS)

            img_rotated = img_resized.rotate(90, expand=True)
            img_final = img_rotated
            # Assume the maximum display size for the image in the GUI window is 500x500 pixels
            max_size = (1000, 1000)

            # Use thumbnail instead of resize, as thumbnail maintains aspect ratio
            img_final.thumbnail(max_size, Image.LANCZOS)

            #img_final = img_rotated.resize((250, 250))
            img_final = ImageTk.PhotoImage(img_final)

            self.img_label.configure(image=img_final)
            self.img_label.image = img_final

            break
        else:
            tk.messagebox.showinfo("End", "No more images.")

    def show_image2(self):
        while self.img_index < len(self.img_files):
            img_path = self.img_files[self.img_index]

            img_name = os.path.basename(self.img_files[self.img_index]).split('.')[0]

            img = Image.open(img_path)

            img_cropped = img.crop((1800, 0, img.width, img.height))

            new_width = img_cropped.width * 1
            new_height = img_cropped.height * 1
            img_resized = img_cropped.resize((new_width, new_height), Image.LANCZOS)

            img_rotated = img_resized.rotate(90, expand=True)
            img_final = img_rotated
            # Assume the maximum display size for the image in the GUI window is 500x500 pixels
            max_size = (1000, 1000)

            # Use thumbnail instead of resize, as thumbnail maintains aspect ratio
            img_final.thumbnail(max_size, Image.LANCZOS)

            #img_final = img_rotated.resize((250, 250))
            img_final = ImageTk.PhotoImage(img_final)

            self.img_label.configure(image=img_final)
            self.img_label.image = img_final

            break
        else:
            tk.messagebox.showinfo("End", "No more images.")

    def goto_image(self):
        index = int(self.index_entry.get())  # Get index from input field and convert to integer
        if 0 <= index < len(self.img_files):  # Check if index is valid
            self.img_index = index  # Update current image index
            self.show_image2()  # Display image
            self.display_data()  # Display data
        else:
            tk.messagebox.showerror("Error", "Invalid index")

    def display_data(self):
        img_name = os.path.basename(self.img_files[self.img_index]).split('.')[0]
        data_row = self.df[self.df['ID'] == img_name]
        if not data_row.empty:
            for col, entry in self.entry_fields.items():
                entry.delete(0, tk.END)  # Clear input field
                entry.insert(0, data_row.iloc[0][col])  # Insert new data
        else:
            for col, entry in self.entry_fields.items():
                entry.delete(0, tk.END)

    def next_image(self):
        self.save_data2()
        if self.img_index < len(self.img_files):
            self.show_image2()
            self.display_data()  # Display data
        else:
            tk.messagebox.showinfo("End", "No more images.")

    def previous_image(self):
        #print(self.img_index)
        self.img_index -= 1
        print(self.img_index)
        if self.img_index < len(self.img_files):
            self.show_image2()
            self.display_data()  # Display data
        else:
            tk.messagebox.showinfo("End", "No images.")

    def save_data(self):
        img_name = os.path.basename(self.img_files[self.img_index]).split('.')[0]  # Get filename without extension
        data = {col: entry.get() for col, entry in self.entry_fields.items()}  # Get data from input fields

        if self.df['ID'].isin([img_name]).any():
            # If ID already exists, update the corresponding row
            for key, value in data.items():
               self.df.loc[self.df['ID'] == img_name, key] = value
        else:
            # If ID does not exist, create a new DataFrame and append
            new_data = pd.DataFrame([[img_name] + list(data.values())], columns=['ID'] + list(data.keys()))
            self.df = pd.concat([self.df, new_data], ignore_index=True)

        self.df.to_csv(self.excel_path, index=False)
        #print(self.img_index)
        self.img_index += 1  # Prepare to display the next image
        print(self.img_index)
        self.show_image()  # Update display

    def save_data2(self):
        img_name = os.path.basename(self.img_files[self.img_index]).split('.')[0]  # Get filename without extension
        data = {col: entry.get() for col, entry in self.entry_fields.items()}  # Get data from input fields

        if self.df['ID'].isin([img_name]).any():
            # If ID already exists, update the corresponding row
            for key, value in data.items():
               self.df.loc[self.df['ID'] == img_name, key] = value
        else:
            # If ID does not exist, create a new DataFrame and append
            new_data = pd.DataFrame([[img_name] + list(data.values())], columns=['ID'] + list(data.keys()))
            self.df = pd.concat([self.df, new_data], ignore_index=True)

        self.df.to_csv(self.excel_path, index=False)
        #print(self.img_index)
        self.img_index += 1  # Prepare to display the next image
        print(self.img_index)
        self.show_image2()  # Update display


def main():
    root = tk.Tk()
    root.title("Image Viewer & Data Entry")
    root.geometry("1100x500")  # Or set other dimensions as needed
    root.resizable(True, True)  # Adjust parameters here as needed

    app = enter_lable(root, 'E:/pythonfile/main-task-image/', 'train_lables.csv')
    root.mainloop()


if __name__ == "__main__":
    main()
